# PhysIR — Crop-Size Study (Kaggle **CPU**, no GPU quota)

Everything is accumulated into a `RESULTS` dict and written to disk at the end:

| file | contents |
|---|---|
| `crop_scale_results.json` | all structured data (machine-readable) |
| `crop_scale_report.md` | human-readable summary |
| `crop_scale_plot.png` | scaling plots |

Download those from `/kaggle/working/` and re-upload them for analysis.

---

## Job 1 ⚠️ Resolution audit — is crop 384 even legal on LOLBlur?

`RandomCropSame` (`data/dataset_reader/datapipeline.py`) **zero-pads** any image smaller than
the crop size:

```python
if img1.shape[1] <= self.size[0] or img1.shape[2] <= self.size[1]:
    img1 = self.pad(img1)          # F.pad(..., (0, mod_pad_w, 0, mod_pad_h)) → ZEROS
```

If LOLBlur frames are shorter than 384px, **every 384 sample carries a black band**. The phase
loss is frequency-domain, so a hard edge injects spurious broadband high-frequency energy —
it would quietly poison a 300-epoch run. (Verified on a simulated 640×360 frame: crop 384 →
6.25% zero pixels.)

**Gate:** proceed with 384 only if `min(H, W) ≥ 384`.

## Job 2 — Phase-loss crop-scale factor, on **real** data

`PhaseEnhancedFrequencyLoss` uses `rfft2(..., norm='backward')` — no forward normalization —
so `|FFT|` magnitudes scale with pixel count `N = H·W`. Inside the loss:

- `amp_loss` → **scales strongly with N**
- `phase_grad` (L1 on *unit* complex-vector differences) → **comparatively stable**, but it
  *mildly decreases* with crop: it is a finite difference between **adjacent frequency bins**,
  and a larger crop gives a finer frequency grid, so neighbouring bins differ less. Expected.

Every other loss is a spatial mean → crop-invariant.

Synthetic 1/f estimate: **factor(256→384) = 1.449** → `phase_weight = 0.20/1.449 = 0.138`.
(Synthetic decomposition: `amp` factor 1.51 ≈ √2.25, `phase_grad` 0.93, amp share 89% —
which reconstructs the 1.449 total.)

This notebook confirms on real LOLBlur and additionally:
- **decomposes phase into `amp` vs `phase_grad`** and checks the parts reconstruct the total
- fits `loss ∝ N^α` for every loss
- measures the image **power-spectrum slope β** (dataset characterisation)
- isolates the `blur` zero-pad border artifact

---
**Accelerator: None (CPU).** A few minutes. No GPU quota.

In [ ]:
# losses/loss.py imports pytorch_msssim at module level, so it's needed even on CPU.
# (torch / torchvision / numpy / PIL / matplotlib are preinstalled on Kaggle.)
!pip install -q pytorch-msssim

## 1. Paths, imports, RESULTS init

In [ ]:
import os, sys, glob, json, math, datetime

# ── EDIT THESE (point at dataset roots; nesting is auto-resolved) ─────────────
CODE_DIR     = '/kaggle/input/datasets/jeffreyboahen/physir-code'
LOLBLUR_ROOT = '/kaggle/input/datasets/jeffreyboahen/lolblur-dataset'
OUT_DIR      = '/kaggle/working'
# ──────────────────────────────────────────────────────────────────────────────

def find_root(base, marker, depth=3):
    """Dir containing `marker`, searching `base` then up to `depth` levels down.
    Kaggle datasets often nest the repo one folder deep (physir-code/PhysIR/)."""
    if os.path.exists(os.path.join(base, marker)):
        return base
    for d in range(1, depth + 1):
        hits = sorted(glob.glob(os.path.join(base, *(['*'] * d), marker)))
        if hits:
            return hits[0][: -len(marker)].rstrip(os.sep)
    return None

CODE_ROOT = find_root(CODE_DIR, os.path.join('losses', 'loss.py'))
if CODE_ROOT is None:
    raise SystemExit(f'Could not find losses/loss.py under {CODE_DIR}')
if CODE_ROOT not in sys.path:
    sys.path.insert(0, CODE_ROOT)

DATA_ROOT = find_root(LOLBLUR_ROOT, os.path.join('train', 'high_sharp_scaled'))
if DATA_ROOT is None:
    raise SystemExit(f'Could not find train/high_sharp_scaled under {LOLBLUR_ROOT}')
GT_DIR  = os.path.join(DATA_ROOT, 'train', 'high_sharp_scaled')
LOW_DIR = os.path.join(DATA_ROOT, 'train', 'low_blur_noise')
assert os.path.isdir(GT_DIR) and os.path.isdir(LOW_DIR)

import torch
import torch.nn.functional as F
import torchvision
from losses.loss import RetinexLoss, BlurAwareGradientLoss, PhaseEnhancedFrequencyLoss
from data.dataset_reader.datapipeline import RandomCropSame

os.makedirs(OUT_DIR, exist_ok=True)

RESULTS = {
    'meta': {
        'timestamp_utc': datetime.datetime.utcnow().isoformat() + 'Z',
        'torch': torch.__version__,
        'torchvision': torchvision.__version__,
        'code_root': CODE_ROOT,
        'data_root': DATA_ROOT,
        'reference_256': {  # measured in real training, Stage 2 epoch 1 (crop 256, eff-batch 8)
            'base': 0.0407, 'phase': 0.0294, 'retinex': 0.0103, 'blur': 0.0059,
        },
        'weights_at_256': {'phase': 0.20, 'retinex': 0.18, 'blur': 0.02},
        'synthetic_estimate': {'phase_factor_384_over_256': 1.4486,
                               'phase_weight_384': 0.1381},
    }
}

print(f'CODE_ROOT : {CODE_ROOT}')
print(f'DATA_ROOT : {DATA_ROOT}')
print(f'torch     : {torch.__version__}')
print('\n✅ losses.loss and datapipeline import cleanly.')

## 2. ⚠️ Resolution audit — the decision gate

In [ ]:
from PIL import Image
from collections import Counter

IMG_EXT = {'.png', '.jpg', '.jpeg', '.bmp'}

def collect(d):
    out = []
    for root, _dirs, files in os.walk(d, followlinks=True):
        for f in sorted(files):
            if os.path.splitext(f)[1].lower() in IMG_EXT:
                out.append(os.path.join(root, f))
    return sorted(out)

gt_paths  = collect(GT_DIR)
low_paths = collect(LOW_DIR)
n_clips = len({os.path.dirname(p) for p in gt_paths})
print(f'GT images: {len(gt_paths)}   LOW images: {len(low_paths)}   clips: {n_clips}')

SAMPLE = min(500, len(gt_paths))
step = max(1, len(gt_paths) // SAMPLE)
sizes = Counter()
min_h = min_w = 10**9
for p in gt_paths[::step][:SAMPLE]:
    with Image.open(p) as im:
        w, h = im.size
    sizes[(w, h)] += 1
    min_h, min_w = min(min_h, h), min(min_w, w)

max_crop = min(min_h, min_w)
print(f'\nSampled {sum(sizes.values())} images. Distinct (W,H): {len(sizes)}')
for (w, h), n in sizes.most_common(8):
    print(f'  {w} x {h}   ({n} imgs)')
print(f'\nmin height={min_h}  min width={min_w}')
print(f'→ largest PAD-FREE square crop = {max_crop}px')

gate = {}
print('\n=== DECISION GATE ===')
for c in (128, 192, 256, 320, 384, 448, 512):
    ok = c <= max_crop
    gate[c] = ok
    print(f'  crop {c:3d}: {"✅ safe" if ok else "❌ WOULD ZERO-PAD every sample"}')

crop384_legal = max_crop >= 384
if crop384_legal:
    print('\n✅ 384 is legal. Stage 4 may use crop 384.')
else:
    print(f'\n❌ 384 is NOT legal — short side is only {max_crop}px.')
    print(f'   Use crop ≤ {max_crop}. If you stay at 256, NO phase rescale is needed.')

RESULTS['audit'] = {
    'n_gt_images': len(gt_paths), 'n_low_images': len(low_paths), 'n_clips': n_clips,
    'sampled': int(sum(sizes.values())),
    'size_histogram': {f'{w}x{h}': int(n) for (w, h), n in sizes.most_common()},
    'min_height': int(min_h), 'min_width': int(min_w),
    'max_padfree_square_crop': int(max_crop),
    'crop_legality': {str(k): bool(v) for k, v in gate.items()},
    'crop384_legal': bool(crop384_legal),
}

## 3. Evidence — what `RandomCropSame` actually does (zero-pad fraction)

In [ ]:
import torchvision.transforms.functional as TF

DEMO_N = 8
demo = {}
print(f'{"crop":>5} {"out shape":>16} {"mean zero-frac":>16}   verdict')
print('-' * 60)
for c in (128, 192, 256, 320, 384, 448, 512):
    fr, shape = [], None
    for p in gt_paths[:DEMO_N]:
        img = TF.to_tensor(Image.open(p).convert('RGB'))
        a, _ = RandomCropSame(c)(img.clone(), img.clone())
        shape = tuple(a.shape)
        fr.append((a.sum(0) == 0).float().mean().item())
    mz = sum(fr) / len(fr)
    demo[c] = {'out_shape': list(shape), 'mean_zero_frac': mz,
               'max_zero_frac': max(fr), 'padded': bool(mz > 0.001)}
    print(f'{c:>5} {str(shape):>16} {mz:>15.2%}   '
          f'{"⚠️ ZERO-PADDED" if mz > 0.001 else "✅ clean"}')

print('\nA non-trivial zero fraction = black band baked into every training sample.')
RESULTS['pad_demo'] = {str(k): v for k, v in demo.items()}

## 4. Loss helpers — including a **decomposition of the phase loss**

The load-bearing claim: inside `PhaseEnhancedFrequencyLoss`, `amp_loss` scales with N while the
phase-gradient half is scale-invariant. We measure each half separately to test that directly.

In [ ]:
retinex_fn = RetinexLoss(weight=1.0)
blur_fn    = BlurAwareGradientLoss(weight=1.0)
phase_fn   = PhaseEnhancedFrequencyLoss(weight=1.0)

def gauss_blur(x, sigma, k=11):
    """REFLECT padding on purpose: zero-padding fabricates a border discontinuity that
    injects broadband HF energy, biasing amp_loss more on small crops."""
    c = torch.arange(k, dtype=x.dtype) - k // 2
    g = torch.exp(-0.5 * (c / sigma) ** 2); g = g / g.sum()
    kern = (g[:, None] * g[None, :]).view(1, 1, k, k).expand(x.shape[1], 1, k, k)
    return F.conv2d(F.pad(x, (k // 2,) * 4, mode='reflect'), kern, groups=x.shape[1])

def make_pred(target, mode):
    """Stand-in for a model output. Prediction QUALITY is held fixed across crops so the
    only thing that varies is crop size."""
    if mode == 'blur':       return gauss_blur(target, 1.0)
    if mode == 'blur+noise': return (gauss_blur(target, 1.0) + 0.02 * torch.randn_like(target)).clamp(0, 1)
    if mode == 'noise':      return (target + 0.05 * torch.randn_like(target)).clamp(0, 1)
    raise ValueError(mode)

@torch.no_grad()
def phase_parts(pred, target):
    """Replicate PhaseEnhancedFrequencyLoss.forward but return amp & phase-grad separately.
    total (weight=1) = 0.5*amp + 0.5*phase_grad."""
    pf = torch.fft.rfft2(pred,   norm='backward')
    tf = torch.fft.rfft2(target, norm='backward')
    mask = phase_fn._high_pass_mask(pf.shape, pf.device)
    amp = F.l1_loss(pf.abs() * mask, tf.abs() * mask)
    pp, tp = phase_fn._phase_unit(pf), phase_fn._phase_unit(tf)
    pdh, pdw = phase_fn._freq_grad(pp)
    tdh, tdw = phase_fn._freq_grad(tp)
    pg = (F.l1_loss(pdh.real * mask[:, :, 1:, :], tdh.real * mask[:, :, 1:, :]) +
          F.l1_loss(pdw.real * mask[:, :, :, 1:], tdw.real * mask[:, :, :, 1:]))
    return amp.item(), pg.item()

def rand_crop_pair(gt, low, size, gen):
    _, H, W = gt.shape
    if H < size or W < size:
        return None, None
    i = torch.randint(0, H - size + 1, (1,), generator=gen).item()
    j = torch.randint(0, W - size + 1, (1,), generator=gen).item()
    return (gt[:, i:i+size, j:j+size].unsqueeze(0), low[:, i:i+size, j:j+size].unsqueeze(0))

@torch.no_grad()
def raw_losses(target, low, mode):
    pred = make_pred(target, mode)
    amp, pg = phase_parts(pred, target)
    return {
        'base':      F.l1_loss(pred, target).item(),
        'retinex':   retinex_fn(pred, target).item(),
        'blur':      blur_fn(pred, low).item(),
        'phase':     phase_fn(pred, target).item(),
        'phase_amp': amp,     # expected: scales with N
        'phase_grad': pg,     # expected: crop-invariant
    }

def mean_std(xs):
    m = sum(xs) / len(xs)
    v = sum((x - m) ** 2 for x in xs) / max(1, len(xs) - 1)
    return m, math.sqrt(v)

print('helpers ready (phase loss decomposed into amp / phase_grad)')

## 5. Crop-scale sweep on real LOLBlur pairs

In [ ]:
N_IMAGES  = 40    # source images
CROPS_PER = 3     # random crops per image per size
MODES     = ['blur', 'blur+noise', 'noise']
SEED      = 42
KEYS      = ['base', 'retinex', 'blur', 'phase', 'phase_amp', 'phase_grad']

CROPS = [c for c in (128, 192, 256, 320, 384, 448, 512) if c <= max_crop]
assert 256 in CROPS, 'need crop 256 as the reference'
print(f'Feasible pad-free crops: {CROPS}   (max_crop={max_crop})\n')

assert len(low_paths) == len(gt_paths), 'GT/LOW count mismatch'
step  = max(1, len(gt_paths) // N_IMAGES)
pairs = list(zip(gt_paths[::step][:N_IMAGES], low_paths[::step][:N_IMAGES]))

torch.manual_seed(SEED)
results = {m: {} for m in MODES}
for mode in MODES:
    for c in CROPS:
        gen = torch.Generator().manual_seed(SEED)   # same crops across modes
        samples = {k: [] for k in KEYS}
        for gp, lp in pairs:
            gt  = TF.to_tensor(Image.open(gp).convert('RGB'))
            low = TF.to_tensor(Image.open(lp).convert('RGB'))
            for _ in range(CROPS_PER):
                t, l = rand_crop_pair(gt, low, c, gen)
                if t is None:
                    continue
                for k, v in raw_losses(t, l, mode).items():
                    samples[k].append(v)
        results[mode][c] = {k: dict(zip(('mean', 'std'), mean_std(v))) for k, v in samples.items()}
        results[mode][c]['n'] = len(samples['base'])
    print(f'  mode={mode:11s} done')

print('\n=== raw loss means (mode=blur) ===')
hdr = f'{"loss":11s}' + ''.join(f'{f"crop {c}":>13s}' for c in CROPS)
print(hdr); print('-' * len(hdr))
for k in KEYS:
    print(f'{k:11s}' + ''.join(f'{results["blur"][c][k]["mean"]:13.6f}' for c in CROPS))

RESULTS['sweep'] = {
    'config': {'n_images': N_IMAGES, 'crops_per_image': CROPS_PER,
               'modes': MODES, 'seed': SEED, 'crops': CROPS},
    'raw': {m: {str(c): results[m][c] for c in CROPS} for m in MODES},
}

## 6. Factors, exponent fit, cross-check, recommended weights

In [ ]:
REF    = 256
TARGET = 384 if 384 in CROPS else max(CROPS)

def mval(mode, c, k):
    return results[mode][c][k]['mean']

def tick(b):
    return 'YES' if b else 'NO '

# ---- factors TARGET/REF, per mode (stability across prediction constructions) ----
print(f'=== factors  crop {TARGET} / crop {REF}  (per mode) ===')
factors = {}
for mode in MODES:
    factors[mode] = {k: mval(mode, TARGET, k) / mval(mode, REF, k) for k in KEYS}
    print(f'  {mode:11s} ' + '  '.join(f'{k}={factors[mode][k]:.4f}' for k in KEYS))

# ---- THE decomposition test ----
# Claim: amp_loss scales strongly with N; phase_grad is *comparatively* stable; and since amp
# dominates the loss, the parts reconstruct the total.
# NOTE: phase_grad is NOT exactly invariant — it mildly DECREASES with crop, because it is a
# finite difference between ADJACENT frequency bins. A larger crop => finer frequency grid =>
# neighbouring bins are closer => smaller phase difference. Expected, not a bug.
print('\n=== DECOMPOSITION TEST (mode=blur) ===')
fa, fg = factors['blur']['phase_amp'], factors['blur']['phase_grad']
a256, g256 = mval('blur', REF, 'phase_amp'), mval('blur', REF, 'phase_grad')
amp_share  = 0.5*a256 / (0.5*a256 + 0.5*g256)
pred_total = (0.5*fa*a256 + 0.5*fg*g256) / (0.5*a256 + 0.5*g256)
meas_total = factors['blur']['phase']
recon_err  = abs(pred_total - meas_total) / meas_total

print(f'  phase_amp  factor = {fa:.4f}   (claim: scales strongly with N)')
print(f'  phase_grad factor = {fg:.4f}   (claim: comparatively stable; mild decrease expected)')
print(f'  amp share of phase loss @{REF} = {amp_share:.1%}')
print(f'  predicted total from parts = {pred_total:.4f}')
print(f'  measured  total            = {meas_total:.4f}   (reconstruction err {recon_err:.2%})')

amp_scales   = fa > 1.20                       # amp clearly grows
grad_stabler = abs(fa - 1) > 3 * abs(fg - 1)   # amp moves much more than grad
parts_recon  = recon_err < 0.01                # parts explain the whole
claim_holds  = amp_scales and grad_stabler and parts_recon

print('')
print(f'  amp scales (>1.20)                : {tick(amp_scales)}')
print(f'  amp moves >> grad (3x)            : {tick(grad_stabler)}   '
      f'|fa-1|={abs(fa-1):.3f} vs |fg-1|={abs(fg-1):.3f}')
print(f'  parts reconstruct total (<1% err) : {tick(parts_recon)}')
verdict = ('CLAIM HOLDS: the crop dependence lives in amp_loss.' if claim_holds
           else 'CLAIM NOT SUPPORTED — investigate before Stage 4.')
print(f'  -> {verdict}')

# ---- exponent fit: loss ∝ N^alpha ----
def fit_alpha(mode, key):
    xs = [math.log(c*c) for c in CROPS]
    ys = [math.log(mval(mode, c, key)) for c in CROPS]
    n = len(xs); mx, my = sum(xs)/n, sum(ys)/n
    a = sum((x-mx)*(y-my) for x, y in zip(xs, ys)) / sum((x-mx)**2 for x in xs)
    ss_res = sum((y-(my+a*(x-mx)))**2 for x, y in zip(xs, ys))
    ss_tot = sum((y-my)**2 for y in ys)
    return a, (1 - ss_res/ss_tot if ss_tot else float('nan'))

print('\n=== scaling exponent   loss ~ N^alpha   (mode=blur) ===')
alphas = {}
for k in ['phase', 'phase_amp', 'phase_grad', 'base', 'retinex', 'blur']:
    a, r2 = fit_alpha('blur', k)
    alphas[k] = {'alpha': a, 'r2': r2}
    print(f'  {k:11s} alpha = {a:+.4f}   R2 = {r2:.4f}')
print('  reference: alpha=0.5 => amp ~ sqrt(N) (natural 1/f images); alpha=0 => crop-invariant')

# ---- cross-check invariance ----
print('\n=== CROSS-CHECK: non-phase losses crop-invariant? (mode=blur) ===')
ok = True
for k in ['base', 'retinex', 'blur']:
    f = factors['blur'][k]
    lo, hi = (0.90, 1.10) if k == 'blur' else (0.95, 1.05)
    good = lo <= f <= hi
    ok &= good
    note = '  (zero-pad border artifact - see section 7)' \
           if (k == 'blur' and not (0.95 <= f <= 1.05)) else ''
    print(f'  {k:8s} factor = {f:.4f}  [{tick(good)}]{note}')
print('  ->', 'CONFIRMED: only phase scales.' if ok else 'INVESTIGATE before Stage 4.')

# ---- recommended weights ----
f_phase = factors['blur']['phase']
new_phase_w = 0.20 / f_phase
print(f'\n=== RECOMMENDED WEIGHTS for crop {TARGET} ===')
if TARGET == 256:
    print('  crop stays 256 -> NO rescale needed. phase=0.20, retinex=0.18, blur=0.02')
else:
    print(f'  measured phase factor ({TARGET}/{REF}) = {f_phase:.4f}')
    print(f'  phase_weight   = 0.20 / {f_phase:.4f} = {new_phase_w:.4f}')
    print('  retinex_weight = 0.18  (unchanged, crop-invariant)')
    print('  blur_weight    = 0.02  (unchanged, crop-invariant)')
print('\n  synthetic 1/f estimate was factor=1.4486 -> phase 0.1381')

RESULTS['analysis'] = {
    'ref_crop': REF, 'target_crop': TARGET,
    'factors_target_over_ref': factors,
    'decomposition': {
        'phase_amp_factor': fa, 'phase_grad_factor': fg,
        'amp_share_at_ref': amp_share,
        'predicted_total_from_parts': pred_total,
        'measured_total': meas_total,
        'reconstruction_err': recon_err,
        'amp_scales': bool(amp_scales),
        'amp_moves_much_more_than_grad': bool(grad_stabler),
        'parts_reconstruct_total': bool(parts_recon),
        'claim_holds': bool(claim_holds),
        'note': ('phase_grad mildly decreases with crop: it is a finite difference between '
                 'adjacent frequency bins, and a larger crop gives a finer frequency grid.'),
    },
    'alphas': alphas,
    'crosscheck_passed': bool(ok),
    'recommended_weights': {'phase': (0.20 if TARGET == 256 else new_phase_w),
                            'retinex': 0.18, 'blur': 0.02},
}

## 7. Why `blur` looks non-invariant — the zero-pad border

`BlurAwareGradientLoss` zero-pads internally (`conv2d padding=5`), fabricating gradients in a
thin border that is a larger fraction of a small crop. Excluding a few px should drive the
factor to ~1.000, proving the loss is intrinsically crop-invariant.

In [ ]:
@torch.no_grad()
def blur_loss_margin(pred, low, m):
    a = blur_fn._sobel(blur_fn._blur(pred))
    b = blur_fn._sobel(low)
    if m:
        a, b = a[:, :, m:-m, m:-m], b[:, :, m:-m, m:-m]
    return F.l1_loss(a, b).item()

margins = [0, 2, 4, 6, 8, 12]
print(f'{"margin":>7} {f"crop {REF}":>12} {f"crop {TARGET}":>12} {"factor":>9}')
print('-' * 44)
margin_tbl = {}
for m in margins:
    vals = {}
    for c in (REF, TARGET):
        gen = torch.Generator().manual_seed(7)
        tot, cnt = 0.0, 0
        for gp, lp in pairs[:20]:
            gt  = TF.to_tensor(Image.open(gp).convert('RGB'))
            low = TF.to_tensor(Image.open(lp).convert('RGB'))
            t, l = rand_crop_pair(gt, low, c, gen)
            if t is None:
                continue
            tot += blur_loss_margin(make_pred(t, 'blur'), l, m); cnt += 1
        vals[c] = tot / cnt
    f = vals[TARGET] / vals[REF]
    margin_tbl[m] = {f'crop{REF}': vals[REF], f'crop{TARGET}': vals[TARGET], 'factor': f}
    print(f'{m:>7} {vals[REF]:12.6f} {vals[TARGET]:12.6f} {f:9.4f}')

print('\nFactor → ~1.000 as margin grows ⇒ blur is intrinsically crop-invariant;')
print('the apparent shift is purely a zero-padding border artifact.')
RESULTS['blur_margin'] = {str(k): v for k, v in margin_tbl.items()}

## 8. Image power-spectrum slope — *context* for the exponent α

Descriptive characterisation of the data: fit `P(f) ∝ f^-β` on the radially-averaged power
spectrum. Natural images have **β ≈ 2** (amplitude `|F| ∝ f^-1`, i.e. "1/f").

> ⚠️ **β/2 is NOT the same quantity as α.** `β/2` is the *frequency* exponent of the amplitude
> spectrum; `α` is the exponent of how `amp_loss` scales with *pixel count* `N`. They are not
> equal and should not be compared numerically. What we can say empirically: images with
> β ≈ 2 produce `amp_loss ∝ N^~0.5`. β is reported here to characterise the dataset (and to
> confirm LOLBlur's GT behaves like natural imagery), not to derive α.

In [ ]:
@torch.no_grad()
def radial_beta(img, nbins=48):
    """Fit P(f) ∝ f^-beta on the radially-averaged power spectrum of a grayscale crop."""
    x = img.mean(1, keepdim=True)                    # (1,1,H,W)
    P = (torch.fft.fft2(x).abs() ** 2).squeeze()
    H, W = P.shape
    fy = torch.fft.fftfreq(H).view(-1, 1)
    fx = torch.fft.fftfreq(W).view(1, -1)
    r  = (fy ** 2 + fx ** 2).sqrt()
    edges = torch.linspace(0.02, 0.45, nbins + 1)    # skip DC and Nyquist corners
    fs, ps = [], []
    for i in range(nbins):
        m = (r >= edges[i]) & (r < edges[i + 1])
        if m.sum() > 8:
            fs.append(((edges[i] + edges[i+1]) / 2).item())
            ps.append(P[m].mean().item())
    xs = [math.log(f) for f in fs]; ys = [math.log(p) for p in ps]
    n = len(xs); mx, my = sum(xs)/n, sum(ys)/n
    slope = sum((a-mx)*(b-my) for a, b in zip(xs, ys)) / sum((a-mx)**2 for a in xs)
    return -slope   # beta

betas_gt, betas_low = [], []
gen = torch.Generator().manual_seed(3)
for gp, lp in pairs[:20]:
    gt  = TF.to_tensor(Image.open(gp).convert('RGB'))
    low = TF.to_tensor(Image.open(lp).convert('RGB'))
    t, l = rand_crop_pair(gt, low, REF, gen)
    if t is None:
        continue
    betas_gt.append(radial_beta(t)); betas_low.append(radial_beta(l))

bg, bgs = mean_std(betas_gt)
bl, bls = mean_std(betas_low)
print('=== power-spectrum slope  P(f) ∝ f^-β  (crop 256) ===')
print(f'  GT  β = {bg:.3f} ± {bgs:.3f}   (natural images ≈ 2.0 ⇒ amplitude ~ 1/f)')
print(f'  LOW β = {bl:.3f} ± {bls:.3f}')
natural = 1.5 <= bg <= 2.6
print(f'  → GT {"behaves like natural imagery ✅" if natural else "is atypical ⚠️"}')

print('\n  NOTE: β/2 is the FREQUENCY exponent of |F|; α is the N-SCALING exponent of amp_loss.')
print('  They are different quantities — do not equate them numerically.')
print(f'  Empirically, β≈2 imagery yields amp_loss ∝ N^~0.5.')
print(f'  measured: phase_amp α = {alphas["phase_amp"]["alpha"]:.3f}  '
      f'(√N would be 0.500)')

# intensity stats — dataset characterization
gt_means, low_means = [], []
for gp, lp in pairs[:20]:
    gt_means.append(TF.to_tensor(Image.open(gp).convert('RGB')).mean().item())
    low_means.append(TF.to_tensor(Image.open(lp).convert('RGB')).mean().item())
gm, gms = mean_std(gt_means); lm, lms = mean_std(low_means)
print(f'\n  mean intensity  GT={gm:.4f}±{gms:.4f}   LOW={lm:.4f}±{lms:.4f}  '
      f'(LOW/GT = {lm/gm:.3f})')

RESULTS['spectrum'] = {
    'beta_gt_mean': bg, 'beta_gt_std': bgs,
    'beta_low_mean': bl, 'beta_low_std': bls,
    'gt_is_natural_like': bool(natural),
    'measured_phase_amp_alpha': alphas['phase_amp']['alpha'],
    'note': ('beta/2 (frequency exponent of |F|) is NOT the same quantity as alpha '
             '(N-scaling exponent of amp_loss); reported for dataset characterisation only.'),
    'intensity': {'gt_mean': gm, 'gt_std': gms, 'low_mean': lm, 'low_std': lms,
                  'low_over_gt': lm / gm},
}

## 9. Plots

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 4.6))

ax = axes[0]
for k, mk in [('phase', 'o'), ('phase_amp', 'v'), ('phase_grad', '^'),
              ('base', 's'), ('retinex', 'P'), ('blur', 'd')]:
    v = [mval('blur', c, k) / mval('blur', REF, k) for c in CROPS]
    ax.plot(CROPS, v, marker=mk, label=k)
ax.axhline(1.0, color='k', ls='--', lw=0.8, alpha=0.6)
ax.axvline(REF, color='gray', ls=':', lw=0.8)
ax.set_xlabel('crop size (px)'); ax.set_ylabel(f'loss / loss@{REF}')
ax.set_title('Only the amplitude half scales'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1]
Ns = [c*c for c in CROPS]
for k in ['phase', 'phase_amp', 'phase_grad']:
    ax.loglog(Ns, [mval('blur', c, k) for c in CROPS], 'o-', label=k)
p0 = mval('blur', CROPS[0], 'phase_amp')
ax.loglog(Ns, [p0 * (N / Ns[0]) ** 0.5 for N in Ns], 'k--', lw=1, label=r'$N^{0.5}$')
ax.set_xlabel('N = crop² (pixels)'); ax.set_ylabel('raw loss')
ax.set_title(f"phase ∝ N^α  (α={alphas['phase']['alpha']:.3f}, R²={alphas['phase']['r2']:.3f})")
ax.legend(fontsize=8); ax.grid(alpha=0.3, which='both')

ax = axes[2]
ms = sorted(margin_tbl)
ax.plot(ms, [margin_tbl[m]['factor'] for m in ms], 'o-', color='crimson')
ax.axhline(1.0, color='k', ls='--', lw=0.8)
ax.set_xlabel('excluded border margin (px)'); ax.set_ylabel(f'blur factor {TARGET}/{REF}')
ax.set_title('blur: zero-pad border artifact'); ax.grid(alpha=0.3)

plt.tight_layout()
png = os.path.join(OUT_DIR, 'crop_scale_plot.png')
plt.savefig(png, dpi=130, bbox_inches='tight')
plt.show()
print('saved', png)

## 10. Write results to disk (download these)

In [ ]:
json_path = os.path.join(OUT_DIR, 'crop_scale_results.json')
with open(json_path, 'w') as f:
    json.dump(RESULTS, f, indent=2, default=float)

A = RESULTS['analysis']; AU = RESULTS['audit']; D = A['decomposition']; S = RESULTS['spectrum']
L = []
L.append('# PhysIR — Crop-Size Study Results\n')
L.append(f"_Generated {RESULTS['meta']['timestamp_utc']} · torch {RESULTS['meta']['torch']}_\n")

L.append('## 1. Resolution audit (decision gate)\n')
L.append(f"- GT images: **{AU['n_gt_images']}** across **{AU['n_clips']}** clips")
L.append(f"- min height **{AU['min_height']}**, min width **{AU['min_width']}**")
L.append(f"- largest pad-free square crop: **{AU['max_padfree_square_crop']}px**")
L.append(f"- **crop 384 legal: {'YES ✅' if AU['crop384_legal'] else 'NO ❌'}**\n")
L.append('| size (WxH) | count |\n|---|---|')
for k, v in list(AU['size_histogram'].items())[:8]:
    L.append(f'| {k} | {v} |')
L.append('')
L.append('| crop | zero-pad fraction | verdict |\n|---|---|---|')
for c, d in RESULTS['pad_demo'].items():
    L.append(f"| {c} | {d['mean_zero_frac']:.2%} | {'⚠️ padded' if d['padded'] else '✅ clean'} |")
L.append('')

L.append(f"## 2. Scale factors (crop {A['target_crop']} / {A['ref_crop']}, mode=blur)\n")
L.append('| loss | factor |\n|---|---|')
for k, v in A['factors_target_over_ref']['blur'].items():
    L.append(f'| {k} | {v:.4f} |')
L.append('')
L.append('### Stability across prediction constructions\n')
L.append('| mode | phase factor |\n|---|---|')
for m, d in A['factors_target_over_ref'].items():
    L.append(f"| {m} | {d['phase']:.4f} |")
L.append('')

L.append('### Decomposition test (the load-bearing claim)\n')
L.append(f"- `phase_amp` factor  = **{D['phase_amp_factor']:.4f}** (scales with N)")
L.append(f"- `phase_grad` factor = **{D['phase_grad_factor']:.4f}** "
         f"(mild decrease expected — finer frequency grid)")
L.append(f"- amp share of phase loss @{A['ref_crop']}: **{D['amp_share_at_ref']:.1%}**")
L.append(f"- parts reconstruct total: predicted {D['predicted_total_from_parts']:.4f} vs "
         f"measured {D['measured_total']:.4f} (err {D['reconstruction_err']:.2%})")
L.append(f"- amp scales: {'✅' if D['amp_scales'] else '❌'} · "
         f"amp ≫ grad movement: {'✅' if D['amp_moves_much_more_than_grad'] else '❌'} · "
         f"reconstruction: {'✅' if D['parts_reconstruct_total'] else '❌'}")
L.append(f"- **claim holds: {'YES ✅' if D['claim_holds'] else 'NO ⚠️'}**")
L.append(f"- _{D['note']}_\n")

L.append('### Scaling exponents (loss ∝ N^α)\n')
L.append('| loss | α | R² |\n|---|---|---|')
for k, v in A['alphas'].items():
    L.append(f"| {k} | {v['alpha']:+.4f} | {v['r2']:.4f} |")
L.append('')

L.append('## 3. Power-spectrum slope (dataset characterisation)\n')
L.append(f"- GT β = **{S['beta_gt_mean']:.3f} ± {S['beta_gt_std']:.3f}** (natural ≈ 2.0)")
L.append(f"- LOW β = {S['beta_low_mean']:.3f} ± {S['beta_low_std']:.3f}")
L.append(f"- GT natural-like: {'YES ✅' if S['gt_is_natural_like'] else 'NO ⚠️'}")
L.append(f"- measured `phase_amp` α = {S['measured_phase_amp_alpha']:.3f} (√N ⇒ 0.500)")
L.append(f"- _{S['note']}_")
L.append(f"- mean intensity GT {S['intensity']['gt_mean']:.4f} vs LOW "
         f"{S['intensity']['low_mean']:.4f} (LOW/GT {S['intensity']['low_over_gt']:.3f})\n")

L.append('## 4. blur zero-pad border artifact\n')
L.append('| margin (px) | factor |\n|---|---|')
for m, d in RESULTS['blur_margin'].items():
    L.append(f"| {m} | {d['factor']:.4f} |")
L.append('')

W = A['recommended_weights']
L.append('## 5. Recommended Stage 4 weights\n')
L.append(f"- crop: **{A['target_crop']}**")
L.append(f"- phase_weight   = **{W['phase']:.4f}**")
L.append(f"- retinex_weight = **{W['retinex']}**")
L.append(f"- blur_weight    = **{W['blur']}**")
L.append(f"- cross-check passed: {'YES ✅' if A['crosscheck_passed'] else 'NO ⚠️'}")

md_path = os.path.join(OUT_DIR, 'crop_scale_report.md')
with open(md_path, 'w') as f:
    f.write('\n'.join(L) + '\n')

print('WROTE:')
for p in (json_path, md_path, os.path.join(OUT_DIR, 'crop_scale_plot.png')):
    print(f'  {p}   ({os.path.getsize(p)/1024:.1f} KB)' if os.path.exists(p)
          else f'  {p}  MISSING')
print('\nDownload these from the Kaggle output panel and re-upload for analysis.')
print('\n--- report preview ---\n')
print('\n'.join(L[:32]))